In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!cp /content/drive/MyDrive/images.zip -d /content/images.zip
!unzip -q images.zip

In [3]:
!rm /content/images.zip

In [4]:
import os
A=os.listdir('/content')
A

['.config', 'drive', 'images', 'sample_data']

In [58]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

['/content/images/C_1_1_1.raw', '/content/images/C_1_1_16.raw', '/content/images/C_1_1_31.raw', '/content/images/C_1_1_46.raw', '/content/images/C_1_1_61.raw', '/content/images/C_1_1_76.raw', '/content/images/C_1_1_91.raw', '/content/images/C_1_1_106.raw', '/content/images/C_1_1_121.raw', '/content/images/C_1_1_136.raw', '/content/images/C_1_1_151.raw', '/content/images/C_1_1_166.raw', '/content/images/C_1_1_181.raw', '/content/images/C_1_1_196.raw', '/content/images/C_1_1_211.raw', '/content/images/C_1_1_226.raw', '/content/images/C_1_1_241.raw', '/content/images/C_1_1_256.raw', '/content/images/C_1_1_271.raw', '/content/images/C_1_1_286.raw', '/content/images/C_1_1_301.raw', '/content/images/C_1_16_1.raw', '/content/images/C_1_16_16.raw', '/content/images/C_1_16_31.raw', '/content/images/C_1_16_46.raw', '/content/images/C_1_16_61.raw', '/content/images/C_1_16_76.raw', '/content/images/C_1_16_91.raw', '/content/images/C_1_16_106.raw', '/content/images/C_1_16_121.raw', '/content/images

np.str_('/content/images/C_1_1_1.raw')

In [59]:
path1=np.array(path1)
data=np.array(data)

In [60]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [61]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

,imgpath,Porosity,throat radius,pore radius,pore_connection_number,pore shape factor
0,/content/images/C_1_1_1.raw,0.200016,0.000009,0.000013,1.92857,0.037024
1,/content/images/C_1_1_16.raw,0.211707,0.000008,0.000012,2.25000,0.036642
2,/content/images/C_1_1_31.raw,0.233293,0.000008,0.000013,1.96875,0.038013
3,/content/images/C_1_1_46.raw,0.264409,0.000009,0.000014,2.18182,0.039828
4,/content/images/C_1_1_61.raw,0.277295,0.000009,0.000014,2.27273,0.038306
...,...,...,...,...,...,...
9256,/content/images/C_301_301_241.raw,0.136927,0.000008,0.000013,2.35443,0.036514
9257,/content/images/C_301_301_256.raw,0.136191,0.000008,0.000013,2.12658,0.036236
9258,/content/images/C_301_301_271.raw,0.147490,0.000008,0.000014,2.17722,0.033896
9259,/content/images/C_301_301_286.raw,0.161413,0.000008,0.000013,2.18987,0.035108


In [ ]:
df_new_two=df_new.iloc[::2,:]
df_new_two

In [62]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data1=np.array(df_new_two)
scaler = StandardScaler()
data1= scaler.fit_transform(data1[:,1:])
print(data1)
print(type(data1))
data1.shape

[[ 0.11455659  0.58493673  0.28969785 -0.82815281 -0.22691552]
 [ 0.27058416 -0.34306756 -0.15901557 -0.0260099  -0.34020621]
 [ 0.55866997 -0.05166937  0.52032542 -0.72788183  0.06680977]
 ...
 [-0.58645309 -0.39830083  0.62718527 -0.20763563 -1.1561657 ]
 [-0.40063734  0.06434896  0.10229779 -0.17606699 -0.79601231]
 [-0.15552552  0.3087704   1.25751387 -0.71308325 -2.46407987]]
<class 'numpy.ndarray'>


(7921, 5)

In [ ]:
df=pd.DataFrame({'imgpath':df_new_two['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.25,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img_2

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
def Data_generator(df, batch_size, augment=False):
  for start in range(0, df.shape[0], batch_size):
      x_batch_img = []
      y_batch=[]
      end = min(start + batch_size, df.shape[0])

      for idd in range(start,end):
          img = open(np.array(df.imgpath)[idd],'rb').read()
          img_1=np.frombuffer(img,dtype=np.uint8)
          img_1=img_1.reshape(100,100,100,1)
          Porosity=np.array(df.Porosity)[idd]
          throat_radius=np.array(df['throat radius'])[idd]
          pore_radius=np.array(df['pore radius'])[idd]
          pore_connection_number=np.array(df.pore_connection_number)[idd]
          pore_shape_factor=np.array(df['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          x_batch_img.append(img_1)
          y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      print(x_batch_img.shape)
      # x_batch_img = tf.squeeze(np.array(x_batch_img), axis=-1)
      y_batch= np.array(y_batch)
      y_batch=y_batch.reshape(-1,5)

      datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.2,height_shift_range=0.2,horizontal_flip=True)
      # datagen.fit(x_batch_img)
      data = datagen.flow(x_batch_img, y_batch, batch_size=32, shuffle=True)
      X = []
      while True:
        try:
          X.append(data.next())
        except:
          break
      # print(type(data))
    #y_batch = to_categorical(y_batch,5)
      yield np.array(X)

ImportError: cannot import name 'ImageDataGenerator' from 'keras.preprocessing.image' (/usr/local/lib/python3.10/dist-packages/keras/api/preprocessing/image/__init__.py)

In [ ]:
pip install keras.preprocessing.image.ImageDataGenerator

ERROR: Could not find a version that satisfies the requirement keras.preprocessing.image.ImageDataGenerator (from versions: none)
ERROR: No matching distribution found for keras.preprocessing.image.ImageDataGenerator


In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import cv2
def apply_augmentation(image):
    final_augmented = []
    final_augmented.append(image)
    for i in range(3):
        rotation_angle = (i+1) * 90
        rotation_matrix = cv2.getRotationMatrix2D((100 // 2, 100 // 2), rotation_angle, 1.0)
        rotated_slices = [cv2.warpAffine(slice_2d, rotation_matrix, (100, 100)) for slice_2d in image]
        augmented_image = np.stack(rotated_slices, axis=0)
        final_augmented.append(augmented_image)
    for j in range(2):
        flipped_slices = [cv2.flip(slice_2d, j) for slice_2d in image]
        augmented_image_2 = np.stack(flipped_slices, axis=0)
        final_augmented.append(augmented_image_2)
    for k in range(2):
        rotation_matrix90 = cv2.getRotationMatrix2D((100 // 2, 100 // 2), 90, 1.0)
        rotated_slices90 = [cv2.warpAffine(slice_2d, rotation_matrix90, (100, 100)) for slice_2d in image]
        augmented_image90 = np.stack(rotated_slices90, axis=0)
        flipped_rotate_slices = [cv2.flip(slice_2d, k) for slice_2d in augmented_image90]
        augmented_image90_f =  np.stack(flipped_rotate_slices, axis=0)
        final_augmented.append(augmented_image90_f)

    return final_augmented


In [ ]:
x = np.zeros((5,5,5,1))
x2 = apply_augmentation(x)
len(x2)

8

In [ ]:
#این دیتا جنریتور است

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
def Data_generator(df, batch_size, augment=False):
  for start in range(0, df.shape[0], batch_size):
      x_batch_img = []
      y_batch = []
      end = min(start + batch_size, df.shape[0])

      for idd in range(start,end):
          img = open(np.array(df.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(df.Porosity)[idd]
          throat_radius=np.array(df['throat radius'])[idd]
          pore_radius=np.array(df['pore radius'])[idd]
          pore_connection_number=np.array(df.pore_connection_number)[idd]
          pore_shape_factor=np.array(df['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)




      yield x_batch_img, y_batch

In [ ]:
#from keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
class Data_generator(tf.keras.utils.Sequence):
  def __init__(self, data, batch_size=4, dim=(100,100,100), channels=1, shuffle=False, augment=False):
    self.data = data
    self.batch_size=batch_size
    self.dim=dim
    self.channels=channels
    self.shuffle=shuffle
    self.on_epoch_end()

  def __len__(self):
    return int(np.floor(len(self.data) / self.batch_size))

  def __getitem__(self, index):
    indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
    # Generate data
    X, y = self.__data_generation(indexes)
    return X, y

  def on_epoch_end(self):

    'Updates indexes after each epoch'
    self.indexes = np.arange(len(self.data))
    if self.shuffle == True:
        np.random.shuffle(self.indexes)

  def __data_generation(self, indexes):
      x_batch_img = []
      y_batch = []

      for idd in indexes:
          img = open(np.array(self.data.imgpath)[idd],'rb').read()
          img_1 = np.frombuffer(img,dtype=np.uint8)
          #img_1 = img_1.reshape(100,100,100,1)
          img_1 = img_1.reshape(100,100,100)
          #img_1 = apply_augmentation(img_1)
          img_1 = [each_image.reshape(100,100,100,1) for each_image in img_1]
          x_batch_img.extend(img_1)
          Porosity=np.array(self.data.Porosity)[idd]
          throat_radius=np.array(self.data['throat radius'])[idd]
          pore_radius=np.array(self.data['pore radius'])[idd]
          pore_connection_number=np.array(self.data.pore_connection_number)[idd]
          pore_shape_factor=np.array(self.data['pore shape factor'])[idd]
          y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
          y=np.array([y_1])
          for i in range(8):
            y_batch.append(y)


      x_batch_img = np.array(x_batch_img)
      y_batch = np.array(y_batch)
      y_batch = y_batch.reshape(-1,5)

      return x_batch_img, y_batch

In [ ]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [38]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield (x_batch_img_2,)

In [ ]:
df_train.shape

(2674, 6)

In [ ]:
batch_size=4
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
batch_size=10
#train_gen= Data_generator(df_train,batch_size)
#valid_gen= Data_generator(df_valid,batch_size)
# train_gen_pre=Data_predict_generator(df_train,batch_size)
# valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen = Data_generator(df_test,batch_size)
# test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
#ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
#nbatches_train=math.ceil(df_train.shape[0]/batch_size)
#nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv3d_5 (Conv3D)                    │ (None, 100, 100, 100, 16)   │           5,504 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 100, 100, 100, 16)   │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_5 (MaxPooling3D)       │ (None, 50, 50, 50, 16)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_6 (Conv3D)                    │ (None, 50, 50, 50, 32)      │          64,032 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 50, 50, 50, 32)      │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_6 (MaxPooling3D)       │ (None, 25, 25, 25, 32)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_7 (Conv3D)                    │ (None, 25, 25, 25, 64)      │          55,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_9                │ (None, 25, 25, 25, 64)      │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_7 (MaxPooling3D)       │ (None, 12, 12, 12, 64)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_8 (Conv3D)                    │ (None, 12, 12, 12, 128)     │         221,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 12, 12, 12, 128)     │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_8 (MaxPooling3D)       │ (None, 6, 6, 6, 128)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_9 (Conv3D)                    │ (None, 6, 6, 6, 256)        │         884,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 6, 6, 6, 256)        │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_9 (MaxPooling3D)       │ (None, 3, 3, 3, 256)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 6912)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1024)                │       7,078,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_12               │ (None, 1024)                │           4,096 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 8,845,605 (33.74 MB)

 Trainable params: 8,841,541 (33.73 MB)

 Non-trainable params: 4,064 (15.88 KB)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/training_model_weights", exist_ok=True)
ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/training_model_weights/weights.{epoch:03d}.keras')
#cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/Training_193.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
history1 = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger])

Epoch 1/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1069s 1s/step - loss: 2.7859 - mse: 2.7859 - val_loss: 11.6688 - val_mse: 11.6688
Epoch 2/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1012s 1s/step - loss: 0.8797 - mse: 0.8797 - val_loss: 2.9453 - val_mse: 2.9453
Epoch 3/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1037s 2s/step - loss: 0.6029 - mse: 0.6029 - val_loss: 0.5320 - val_mse: 0.5320
Epoch 4/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1041s 2s/step - loss: 0.5459 - mse: 0.5459 - val_loss: 531.4863 - val_mse: 531.4863
Epoch 5/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 987s 1s/step - loss: 0.5241 - mse: 0.5241 - val_loss: 316.8066 - val_mse: 316.8066
Epoch 6/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1038s 2s/step - loss: 0.5147 - mse: 0.5147 - val_loss: 1.3417 - val_mse: 1.3417
Epoch 7/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 973s 1s/step - loss: 0.4967 - mse: 0.4967 - val_loss: 0.4156 - val_mse: 0.4156
Epoch 8/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 973s 1s/step - loss: 0.4756 - mse: 0.4756 - val_loss: 0.4295 - val_mse: 0.4295
Epoch 9/200
668/668 ━━━━━━━━━━━━━━━━━

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.024.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=24)

Epoch 25/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1113s 2s/step - loss: 0.3248 - mse: 0.3248 - val_loss: 0.4410 - val_mse: 0.4410
Epoch 26/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1073s 2s/step - loss: 0.3205 - mse: 0.3205 - val_loss: 0.2262 - val_mse: 0.2262
Epoch 27/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 2s/step - loss: 0.3229 - mse: 0.3229 - val_loss: 0.2389 - val_mse: 0.2389
Epoch 28/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1018s 2s/step - loss: 0.3276 - mse: 0.3276 - val_loss: 0.2954 - val_mse: 0.2954
Epoch 29/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1017s 2s/step - loss: 0.3333 - mse: 0.3333 - val_loss: 0.3249 - val_mse: 0.3249
Epoch 30/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1097s 2s/step - loss: 0.3028 - mse: 0.3028 - val_loss: 0.2605 - val_mse: 0.2605
Epoch 31/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1102s 2s/step - loss: 0.3168 - mse: 0.3168 - val_loss: 0.2549 - val_mse: 0.2549
Epoch 32/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1018s 2s/step - loss: 0.3105 - mse: 0.3105 - val_loss: 0.2777 - val_mse: 0.2777
Epoch 33/200
668/668 ━━━━━━━━━━━━━━━━

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/training_model_weights/weights.035.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=35)

Epoch 36/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1072s 1s/step - loss: 0.2971 - mse: 0.2971 - val_loss: 0.2065 - val_mse: 0.2065
Epoch 37/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1016s 1s/step - loss: 0.2927 - mse: 0.2927 - val_loss: 0.2113 - val_mse: 0.2113
Epoch 38/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 982s 1s/step - loss: 0.2952 - mse: 0.2952 - val_loss: 0.2086 - val_mse: 0.2086
Epoch 39/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1035s 2s/step - loss: 0.3068 - mse: 0.3068 - val_loss: 0.2473 - val_mse: 0.2473
Epoch 40/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 981s 1s/step - loss: 0.3054 - mse: 0.3054 - val_loss: 0.2314 - val_mse: 0.2314
Epoch 41/200
301/668 ━━━━━━━━━━━━━━━━━━━━ 8:09 1s/step - loss: 0.3046 - mse: 0.3046

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/training_model_weights/weights.040.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=40)

Epoch 41/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1063s 1s/step - loss: 0.2897 - mse: 0.2897 - val_loss: 0.2223 - val_mse: 0.2223
Epoch 42/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1073s 2s/step - loss: 0.3196 - mse: 0.3196 - val_loss: 0.2133 - val_mse: 0.2133
Epoch 43/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 976s 1s/step - loss: 0.2985 - mse: 0.2985 - val_loss: 0.2420 - val_mse: 0.2420
Epoch 44/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.3030 - mse: 0.3030

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.051.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=51)

Epoch 52/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1083s 2s/step - loss: 0.2971 - mse: 0.2971 - val_loss: 0.2021 - val_mse: 0.2021
Epoch 53/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 980s 1s/step - loss: 0.2944 - mse: 0.2944 - val_loss: 0.2386 - val_mse: 0.2386
Epoch 54/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 981s 1s/step - loss: 0.2802 - mse: 0.2802 - val_loss: 0.2107 - val_mse: 0.2107
Epoch 55/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 957s 1s/step - loss: 0.2935 - mse: 0.2935 - val_loss: 0.2089 - val_mse: 0.2089
Epoch 56/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 956s 1s/step - loss: 0.2895 - mse: 0.2895 - val_loss: 0.3183 - val_mse: 0.3183
Epoch 57/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1010s 2s/step - loss: 0.2995 - mse: 0.2995 - val_loss: 0.1951 - val_mse: 0.1951
Epoch 58/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 988s 1s/step - loss: 0.2864 - mse: 0.2864 - val_loss: 0.2259 - val_mse: 0.2259
Epoch 59/200
117/668 ━━━━━━━━━━━━━━━━━━━━ 11:54 1s/step - loss: 0.3369 - mse: 0.3369

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.064.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=64)

Epoch 65/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1039s 1s/step - loss: 0.2840 - mse: 0.2840 - val_loss: 0.2104 - val_mse: 0.2104
Epoch 66/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 970s 1s/step - loss: 0.2865 - mse: 0.2865 - val_loss: 0.2204 - val_mse: 0.2204
Epoch 67/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1036s 2s/step - loss: 0.2921 - mse: 0.2921 - val_loss: 0.1912 - val_mse: 0.1912
Epoch 68/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.2861 - mse: 0.2861

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.075.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=75)

Epoch 76/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1109s 2s/step - loss: 0.2798 - mse: 0.2798 - val_loss: 0.1922 - val_mse: 0.1922
Epoch 77/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1013s 2s/step - loss: 0.2907 - mse: 0.2907 - val_loss: 0.3053 - val_mse: 0.3053
Epoch 78/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 2s/step - loss: 0.2815 - mse: 0.2815 - val_loss: 0.2028 - val_mse: 0.2028
Epoch 79/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1043s 2s/step - loss: 0.2831 - mse: 0.2831 - val_loss: 0.1925 - val_mse: 0.1925
Epoch 80/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.2831 - mse: 0.2831

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.112.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=112)

Epoch 113/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1107s 2s/step - loss: 0.2762 - mse: 0.2762 - val_loss: 0.1877 - val_mse: 0.1877
Epoch 114/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1124s 2s/step - loss: 0.2882 - mse: 0.2882 - val_loss: 0.1981 - val_mse: 0.1981
Epoch 115/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1012s 2s/step - loss: 0.2707 - mse: 0.2707 - val_loss: 0.1944 - val_mse: 0.1944
Epoch 116/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 2s/step - loss: 0.2801 - mse: 0.2801 - val_loss: 0.2418 - val_mse: 0.2418
Epoch 117/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1011s 2s/step - loss: 0.2778 - mse: 0.2778 - val_loss: 0.1861 - val_mse: 0.1861
Epoch 118/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1096s 2s/step - loss: 0.2723 - mse: 0.2723 - val_loss: 0.1980 - val_mse: 0.1980
Epoch 119/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1047s 2s/step - loss: 0.2760 - mse: 0.2760 - val_loss: 0.1895 - val_mse: 0.1895
Epoch 120/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 2s/step - loss: 0.2646 - mse: 0.2646 - val_loss: 0.1860 - val_mse: 0.1860
Epoch 121/200
668/668 ━━━━━━━━

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.134.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=134)

Epoch 135/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1080s 2s/step - loss: 0.2827 - mse: 0.2827 - val_loss: 0.1834 - val_mse: 0.1834
Epoch 136/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 997s 1s/step - loss: 0.2884 - mse: 0.2884 - val_loss: 0.1891 - val_mse: 0.1891
Epoch 137/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1041s 1s/step - loss: 0.2874 - mse: 0.2874 - val_loss: 0.2446 - val_mse: 0.2446
Epoch 138/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 1s/step - loss: 0.2860 - mse: 0.2860 - val_loss: 0.2104 - val_mse: 0.2104
Epoch 139/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1040s 1s/step - loss: 0.2804 - mse: 0.2804 - val_loss: 0.1982 - val_mse: 0.1982
Epoch 140/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 993s 1s/step - loss: 0.2774 - mse: 0.2774 - val_loss: 0.1844 - val_mse: 0.1844
Epoch 141/200
338/668 ━━━━━━━━━━━━━━━━━━━━ 7:25 1s/step - loss: 0.2820 - mse: 0.2820

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/shared/training_model_weights/weights.192.keras')

In [ ]:
history = model.fit(train_gen, epochs=200, verbose=1, validation_data=valid_gen ,callbacks=[ckpt_callback,csvlogger],initial_epoch=192)

Epoch 193/200


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


668/668 ━━━━━━━━━━━━━━━━━━━━ 1098s 2s/step - loss: 0.2839 - mse: 0.2839 - val_loss: 0.1851 - val_mse: 0.1851
Epoch 194/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1003s 1s/step - loss: 0.2655 - mse: 0.2655 - val_loss: 0.1861 - val_mse: 0.1861
Epoch 195/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1045s 2s/step - loss: 0.2674 - mse: 0.2674 - val_loss: 0.1934 - val_mse: 0.1934
Epoch 196/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1002s 1s/step - loss: 0.2839 - mse: 0.2839 - val_loss: 0.1907 - val_mse: 0.1907
Epoch 197/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1056s 2s/step - loss: 0.2808 - mse: 0.2808 - val_loss: 0.1857 - val_mse: 0.1857
Epoch 198/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1002s 1s/step - loss: 0.2895 - mse: 0.2895 - val_loss: 0.1899 - val_mse: 0.1899
Epoch 199/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1042s 1s/step - loss: 0.2687 - mse: 0.2687 - val_loss: 0.1831 - val_mse: 0.1831
Epoch 200/200
668/668 ━━━━━━━━━━━━━━━━━━━━ 1002s 1s/step - loss: 0.2677 - mse: 0.2677 - val_loss: 0.2206 - val_mse: 0.2206


In [ ]:
######################################### predict #####################################

In [ ]:
model= load_model('/content/drive/MyDrive/Sensivity_analysis/Augment/50%dataset/With_augment/training_model_weights/weights.200.keras')

In [ ]:
model.evaluate(test_gen, nbatches_test)

ValueError: When providing `x` as a PyDataset, `y` should not be passed. Instead, the targets should be included as part of the PyDataset.

In [ ]:
######################## activation function #################################

In [9]:
df_new_two=df_new.iloc[::1,:]
df_new_two

,imgpath,Porosity,throat radius,pore radius,pore_connection_number,pore shape factor
0,/content/images/C_1_1_1.raw,0.200016,0.000009,0.000013,1.92857,0.037024
1,/content/images/C_1_1_16.raw,0.211707,0.000008,0.000012,2.25000,0.036642
2,/content/images/C_1_1_31.raw,0.233293,0.000008,0.000013,1.96875,0.038013
3,/content/images/C_1_1_46.raw,0.264409,0.000009,0.000014,2.18182,0.039828
4,/content/images/C_1_1_61.raw,0.277295,0.000009,0.000014,2.27273,0.038306
...,...,...,...,...,...,...
9256,/content/images/C_301_301_241.raw,0.136927,0.000008,0.000013,2.35443,0.036514
9257,/content/images/C_301_301_256.raw,0.136191,0.000008,0.000013,2.12658,0.036236
9258,/content/images/C_301_301_271.raw,0.147490,0.000008,0.000014,2.17722,0.033896
9259,/content/images/C_301_301_286.raw,0.161413,0.000008,0.000013,2.18987,0.035108


In [10]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data1=np.array(df_new_two)
scaler = StandardScaler()
data1= scaler.fit_transform(data1[:,1:])
print(data1)
print(type(data1))
data1.shape

[[ 0.11455659  0.58493673  0.28969785 -0.82815281 -0.22691552]
 [ 0.27058416 -0.34306756 -0.15901557 -0.0260099  -0.34020621]
 [ 0.55866997 -0.05166937  0.52032542 -0.72788183  0.06680977]
 ...
 [-0.58645309 -0.39830083  0.62718527 -0.20763563 -1.1561657 ]
 [-0.40063734  0.06434896  0.10229779 -0.17606699 -0.79601231]
 [-0.15552552  0.3087704   1.25751387 -0.71308325 -2.46407987]]
<class 'numpy.ndarray'>


(7921, 5)

In [11]:
df=pd.DataFrame({'imgpath':df_new_two['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

,imgpath,Porosity,throat radius,pore radius,pore_connection_number,pore shape factor
0,/content/images/C_1_1_1.raw,0.114557,0.584937,0.289698,-0.828153,-0.226916
1,/content/images/C_1_1_16.raw,0.270584,-0.343068,-0.159016,-0.026010,-0.340206
2,/content/images/C_1_1_31.raw,0.558670,-0.051669,0.520325,-0.727882,0.066810
3,/content/images/C_1_1_46.raw,0.973943,0.745227,0.630089,-0.196156,0.606040
4,/content/images/C_1_1_61.raw,1.145919,0.514543,1.057970,0.030714,0.154056
...,...,...,...,...,...,...
9256,/content/images/C_301_301_241.raw,-0.727426,-0.487201,0.223571,0.234600,-0.378288
9257,/content/images/C_301_301_256.raw,-0.737249,-0.264275,0.454180,-0.334010,-0.460952
9258,/content/images/C_301_301_271.raw,-0.586453,-0.398301,0.627185,-0.207636,-1.156166
9259,/content/images/C_301_301_286.raw,-0.400637,0.064349,0.102298,-0.176067,-0.796012


In [12]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.25,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

5347

In [13]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data1=np.array(df_new_two)
scaler = StandardScaler()
data1= scaler.fit_transform(data1[:,1:])
print(data1)
print(type(data1))
data1.shape

[[ 0.11455659  0.58493673  0.28969785 -0.82815281 -0.22691552]
 [ 0.27058416 -0.34306756 -0.15901557 -0.0260099  -0.34020621]
 [ 0.55866997 -0.05166937  0.52032542 -0.72788183  0.06680977]
 ...
 [-0.58645309 -0.39830083  0.62718527 -0.20763563 -1.1561657 ]
 [-0.40063734  0.06434896  0.10229779 -0.17606699 -0.79601231]
 [-0.15552552  0.3087704   1.25751387 -0.71308325 -2.46407987]]
<class 'numpy.ndarray'>


(7921, 5)

In [14]:
df=pd.DataFrame({'imgpath':df_new_two['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

,imgpath,Porosity,throat radius,pore radius,pore_connection_number,pore shape factor
0,/content/images/C_1_1_1.raw,0.114557,0.584937,0.289698,-0.828153,-0.226916
1,/content/images/C_1_1_16.raw,0.270584,-0.343068,-0.159016,-0.026010,-0.340206
2,/content/images/C_1_1_31.raw,0.558670,-0.051669,0.520325,-0.727882,0.066810
3,/content/images/C_1_1_46.raw,0.973943,0.745227,0.630089,-0.196156,0.606040
4,/content/images/C_1_1_61.raw,1.145919,0.514543,1.057970,0.030714,0.154056
...,...,...,...,...,...,...
9256,/content/images/C_301_301_241.raw,-0.727426,-0.487201,0.223571,0.234600,-0.378288
9257,/content/images/C_301_301_256.raw,-0.737249,-0.264275,0.454180,-0.334010,-0.460952
9258,/content/images/C_301_301_271.raw,-0.586453,-0.398301,0.627185,-0.207636,-1.156166
9259,/content/images/C_301_301_286.raw,-0.400637,0.064349,0.102298,-0.176067,-0.796012


In [15]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.25,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

5347

In [16]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [39]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield (x_batch_img_2,)

In [40]:
batch_size=10
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [41]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

179 535 80


In [20]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [21]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='sigmoid',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='sigmoid',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='sigmoid',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='sigmoid',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='sigmoid'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='sigmoid'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 100, 100, 100,  │         5,504 │
│                                 │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 100, 100,  │            64 │
│ (BatchNormalization)            │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 50, 50, 50, 16) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 50, 50, 50, 32) │        64,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 50, 50, 32) │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 25, 25, 25, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 25, 25, 25, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 25, 25, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 12, 12, 12, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 12, 12, 12,     │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 12, 12,     │           512 │
│ (BatchNormalization)            │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_3 (MaxPooling3D)  │ (None, 6, 6, 6, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_4 (Conv3D)               │ (None, 6, 6, 6, 256)   │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 6, 6, 6, 256)   │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_4 (MaxPooling3D)  │ (None, 3, 3, 3, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6912)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     7,078,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,04

 Total params: 8,845,605 (33.74 MB)

 Trainable params: 8,841,541 (33.73 MB)

 Non-trainable params: 4,064 (15.88 KB)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/Sensivity_analysis/activation_function/sigmoid/training_model_weights", exist_ok=True)
ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/Sensivity_analysis/activation_function/sigmoid/training_model_weights/weights.{epoch:03d}.keras', monitor='val_loss')

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/Sensivity_analysis/activation_function/sigmoid/Training_154.log')

In [22]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [23]:
from tensorflow.keras.models import load_model

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/shared/training_model_weights/weights.153.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 153)

Epoch 154/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 335s 560ms/step - loss: 0.1456 - mse: 0.1456 - val_loss: 0.1534 - val_mse: 0.1534
Epoch 155/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 273s 510ms/step - loss: 0.1446 - mse: 0.1446 - val_loss: 0.1877 - val_mse: 0.1877
Epoch 156/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 272s 509ms/step - loss: 0.1447 - mse: 0.1447 - val_loss: 0.1673 - val_mse: 0.1673
Epoch 157/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 272s 509ms/step - loss: 0.1428 - mse: 0.1428 - val_loss: 0.1551 - val_mse: 0.1551
Epoch 158/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 273s 510ms/step - loss: 0.1432 - mse: 0.1432 - val_loss: 0.1516 - val_mse: 0.1516
Epoch 159/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 272s 509ms/step - loss: 0.1436 - mse: 0.1436 - val_loss: 0.1496 - val_mse: 0.1496
Epoch 160/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 272s 509ms/step - loss: 0.1444 - mse: 0.1444 - val_loss: 0.1679 - val_mse: 0.1679
Epoch 161/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 272s 509ms/step - loss: 0.1440 - mse: 0.1440 - val_loss: 0.1674 - val_mse: 0.1674


In [24]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/sigmoid/training_model_weights/weights.200.keras')

In [31]:
model.evaluate(test_gen, steps=nbatches_test)

80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 138ms/step - loss: 0.1482 - mse: 0.1482


[0.14368939399719238, 0.14368939399719238]

In [42]:
y=model.predict(x=test_gen_pre,verbose=1,steps=nbatches_test)

80/80 ━━━━━━━━━━━━━━━━━━━━ 13s 147ms/step


In [43]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [44]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [45]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

0.9747071297076249


In [46]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

0.687160276202462


In [47]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

0.8313891809059888


In [48]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

0.8948444868032113


In [49]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)

0.8847298536329926


In [ ]:
#################### elu #############################

In [50]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='elu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='elu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='elu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='elu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='elu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='elu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='elu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d_5 (Conv3D)               │ (None, 100, 100, 100,  │         5,504 │
│                                 │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 100, 100, 100,  │            64 │
│ (BatchNormalization)            │ 16)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_5 (MaxPooling3D)  │ (None, 50, 50, 50, 16) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_6 (Conv3D)               │ (None, 50, 50, 50, 32) │        64,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 50, 50, 50, 32) │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_6 (MaxPooling3D)  │ (None, 25, 25, 25, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_7 (Conv3D)               │ (None, 25, 25, 25, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 25, 25, 25, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_7 (MaxPooling3D)  │ (None, 12, 12, 12, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_8 (Conv3D)               │ (None, 12, 12, 12,     │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 12, 12, 12,     │           512 │
│ (BatchNormalization)            │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_8 (MaxPooling3D)  │ (None, 6, 6, 6, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_9 (Conv3D)               │ (None, 6, 6, 6, 256)   │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 6, 6, 6, 256)   │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_9 (MaxPooling3D)  │ (None, 3, 3, 3, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6912)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1024)           │     7,078,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 512)            │         2,04

 Total params: 8,845,605 (33.74 MB)

 Trainable params: 8,841,541 (33.73 MB)

 Non-trainable params: 4,064 (15.88 KB)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights", exist_ok=True)
ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights/weights.{epoch:03d}.keras', monitor='val_loss')

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/Sensivity_analysis/activation_function/sigmoid/Training_178.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/shared/training_model_weights/weights.051.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 51)

Epoch 52/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 345s 569ms/step - loss: 0.1412 - mse: 0.1412 - val_loss: 0.1306 - val_mse: 0.1306
Epoch 53/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 290s 542ms/step - loss: 0.1400 - mse: 0.1400 - val_loss: 0.1547 - val_mse: 0.1547
Epoch 54/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 289s 540ms/step - loss: 0.1396 - mse: 0.1396 - val_loss: 0.1368 - val_mse: 0.1368
Epoch 55/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 289s 540ms/step - loss: 0.1392 - mse: 0.1392 - val_loss: 0.1751 - val_mse: 0.1751
Epoch 56/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 289s 540ms/step - loss: 0.1400 - mse: 0.1400 - val_loss: 0.1962 - val_mse: 0.1962
Epoch 57/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 288s 539ms/step - loss: 0.1386 - mse: 0.1386 - val_loss: 0.1431 - val_mse: 0.1431
Epoch 58/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 289s 540ms/step - loss: 0.1360 - mse: 0.1360 - val_loss: 0.1259 - val_mse: 0.1259
Epoch 59/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 290s 541ms/step - loss: 0.1367 - mse: 0.1367 - val_loss: 0.1360 - val_mse: 0.1360
Epoch 60

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights/weights.069.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 69)

Epoch 70/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 334s 558ms/step - loss: 0.1309 - mse: 0.1309 - val_loss: 0.1455 - val_mse: 0.1455
Epoch 71/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 277s 519ms/step - loss: 0.1336 - mse: 0.1336 - val_loss: 0.1659 - val_mse: 0.1659
Epoch 72/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 277s 518ms/step - loss: 0.1303 - mse: 0.1303 - val_loss: 0.1452 - val_mse: 0.1452
Epoch 73/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 277s 517ms/step - loss: 0.1316 - mse: 0.1316 - val_loss: 0.1343 - val_mse: 0.1343
Epoch 74/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 276s 517ms/step - loss: 0.1309 - mse: 0.1309 - val_loss: 0.1214 - val_mse: 0.1214
Epoch 75/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 276s 516ms/step - loss: 0.1302 - mse: 0.1302 - val_loss: 0.1363 - val_mse: 0.1363
Epoch 76/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 276s 517ms/step - loss: 0.1310 - mse: 0.1310 - val_loss: 0.1570 - val_mse: 0.1570
Epoch 77/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 276s 516ms/step - loss: 0.1297 - mse: 0.1297 - val_loss: 0.1259 - val_mse: 0.1259
Epoch 78

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights/weights.089.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 89)

Epoch 90/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 345s 564ms/step - loss: 0.1270 - mse: 0.1270 - val_loss: 0.1305 - val_mse: 0.1305
Epoch 91/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 283s 530ms/step - loss: 0.1272 - mse: 0.1272 - val_loss: 0.1233 - val_mse: 0.1233
Epoch 92/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 530ms/step - loss: 0.1269 - mse: 0.1269 - val_loss: 0.1216 - val_mse: 0.1216
Epoch 93/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 531ms/step - loss: 0.1263 - mse: 0.1263 - val_loss: 0.1217 - val_mse: 0.1217
Epoch 94/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 531ms/step - loss: 0.1273 - mse: 0.1273 - val_loss: 0.1179 - val_mse: 0.1179
Epoch 95/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 285s 532ms/step - loss: 0.1275 - mse: 0.1275 - val_loss: 0.1288 - val_mse: 0.1288
Epoch 96/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 283s 530ms/step - loss: 0.1242 - mse: 0.1242 - val_loss: 0.1410 - val_mse: 0.1410
Epoch 97/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 530ms/step - loss: 0.1265 - mse: 0.1265 - val_loss: 0.1383 - val_mse: 0.1383
Epoch 98

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights/weights.101.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 101)

Epoch 102/200
479/535 ━━━━━━━━━━━━━━━━━━━━ 26s 471ms/step - loss: 0.1251 - mse: 0.1251

In [ ]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/shared/training_model_weights/weights.177.keras')

In [ ]:
history=model.fit(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[ckpt_callback,csvlogger],initial_epoch = 177)

Epoch 178/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 342s 568ms/step - loss: 0.1197 - mse: 0.1197 - val_loss: 0.1150 - val_mse: 0.1150
Epoch 179/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 531ms/step - loss: 0.1204 - mse: 0.1204 - val_loss: 0.1101 - val_mse: 0.1101
Epoch 180/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 530ms/step - loss: 0.1192 - mse: 0.1192 - val_loss: 0.1119 - val_mse: 0.1119
Epoch 181/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 530ms/step - loss: 0.1207 - mse: 0.1207 - val_loss: 0.1148 - val_mse: 0.1148
Epoch 182/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 283s 530ms/step - loss: 0.1190 - mse: 0.1190 - val_loss: 0.1204 - val_mse: 0.1204
Epoch 183/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 283s 530ms/step - loss: 0.1209 - mse: 0.1209 - val_loss: 0.1108 - val_mse: 0.1108
Epoch 184/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 284s 530ms/step - loss: 0.1192 - mse: 0.1192 - val_loss: 0.1099 - val_mse: 0.1099
Epoch 185/200
535/535 ━━━━━━━━━━━━━━━━━━━━ 283s 530ms/step - loss: 0.1179 - mse: 0.1179 - val_loss: 0.1131 - val_mse: 0.1131


In [51]:
model = load_model('/content/drive/MyDrive/Sensivity_analysis/activation_function/elu/training_model_weights/weights.200.keras')

In [52]:
model.evaluate(test_gen, steps=nbatches_test)

80/80 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - loss: 0.1131 - mse: 0.1131


[0.10501433163881302, 0.10501433163881302]

In [53]:
y=model.predict(x=test_gen_pre,verbose=1,steps=nbatches_test)

80/80 ━━━━━━━━━━━━━━━━━━━━ 14s 151ms/step


In [54]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [55]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [56]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

-0.870163362399925


In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

0.687160276202462


In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

0.8313891809059888


In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

0.8948444868032113


In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)

0.8847298536329926
